**Project:** Data Mining II (2025/26)

**Group Number:** 12

**Members:**
- Beatriz Boura - 20250272
- Dinis Gaspar - 20221869
- Margarida Cruz - 20221929

**Project Overview**

In recent years, nonprofit organizations have faced a growing challenge: while charitable causes have multiplied, public tolerance for repeated, generic solicitations has significantly decreased, often leading to donor fatigue and long-term disengagement. To address this issue, the Civic Support Alliance (CSA)—a federation representing multiple humanitarian and social aid programs—seeks to modernize its fundraising strategy. Rather than launching blanket campaigns across their entire database, the organization aims to transition to a highly targeted approach. The goal is to maximize operational efficiency and maintain donor respect by contacting fewer, but more receptive, individuals. 

As data scientists, our team has been tasked with building a predictive machine learning system using historical demographic, interaction, and donation data accumulated from past campaigns. The primary objective is to accurately answer a fundamental question: Will this person donate if contacted? 

**Notebook Introduction**

In this notebook, we'll showcase and explain the pipeline and tools we have developed for the modeling stage of our project. They will be stored and then pulled from the utils_modeling.py for use. The goal of this process was to use scikit-learn's built pipeline capabilities to design pipeline objects such that predicting the test set would require as little preparation before hand as possible. 

In [ ]:
%cd ..
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold, train_test_split, TunedThresholdClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid, cross_validate
from tqdm.auto import tqdm
import pickle
import warnings
warnings.filterwarnings("ignore")

c:\Users\dinis\OneDrive\Ambiente de Trabalho\Faculdade - MGI-BI\1º ano\2º Semestre\Data Mining II\Project\DM2_Project


## General

To develop our pipeline we will use scikit-learn's Pipeline capabilities which enables us to combine various transformers within a single Pipeline object and design a streamlined end-to-end pipeline that can be used with simple .fit() and .predict() methods, as such we have developed some custom transformers which will help us in completing the full pipeline, some of which are problem specific and can't be transfered, but also some which can be re-used with totally different problems.

In the development of our pipeline we will split data into categorical columns and numerical columns, so that different techniques can be used for imputation, feature selection, etc.

As such it is important to note that for the development and implementation of our pipeline whenever the processing of categorical and numerical features is split, ordinal and binary features will be considered numerical. For ordinal features this is done since the values have order and similarities/dissimilarities can be infered simply across categories based on the encoded value. For binary values, this decision comes from the fact that an imputed value of 0.7 can actually be more useful than a mode imputed value of 1, for example. Thus, this distinction aims to extract as much information as possible from these categorical columns. This leaves URBANICITY, RECENCY_STATUS_96NK and DONOR_GENDER as the only variables considered as categorical, and for these variables mode imputation is actual a suitable, yet simple option, since their distributions are far from uniform.

As far as our model testing methodology is concerned, we will have a notebook for each type of model (Tree models, ensemble, Instance-based, etc.) which will allow us to keep everything organized and separate. Additionally within each notebook we'll test general parameters, using a hold-out train-test split, to look for regions of parameter values to use for more advanced parameter search methods, such as grid search. When performing parameter searching, we will use a StratifiedKFold with 5 folds, as we believe this provides us with stable results without massively increasing the computational complexity and runtime. We will instatiate that StratifiedKFold below and we will then export so that it can be used for each testing notebook.

In [2]:
model_testing_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=23)
with open('Files/Pickle Files/model_testing_skf.pkl', 'wb') as file:
    pickle.dump(model_testing_skf, file)

## DataCleaner
The DataCleaner transformer serves as the first step in our pipeline, and as such focuses on handling data quality issues before proceeding with further preprocessing steps, as such it performs the following functions, based on insights extracted during data exploration:
+ Drop the CONTROL_NUMBER identifier column.
+ Replace the '?' values present in a couple of the variables with null values and correct the datatype of SES to be float since default pandas integer datatypes don't accept null values.
+ Replace the long decimal values which we identified as being incorrect with null values, using the check_decimals support function, with a threshold to remove any values with more than 3 decimal places, which we identified as not being genuine.
+ Get the absolute value of any remaining negative values, since those, from our analysis, seemed like valid values with the sign flipped.
+ Transform the HOME_OWNER into a binary 0-1 encoded flag variable.
+ Correct the values of RECENT_STAR_STATUS which are greater than 1 to be 1, as despite the fact that these may be valid values resulting from an undocumented evolution of the column from an indicator to a count, the frequency of values greater than 1 is too sparse to justify maintaining the variable as is, it's much better to transform it to ensure it's an indicator flag variable.
+ As more a future proofing measure for the categorical variables transform any unknown value, based on the variable description and characteristics, into a missing value.

Strictly speaking, the DataCleaner is stateless and not problem transferable and it's impact doesn't change or have to be re-run from fold to fold of a cross-validation process, and as such this step could conceptually be extracted from the Pipeline object and run once on the full training set, however it's inclusion facilitates the process of predicting the test set since the class handles the data issues from within the pipeline. Additionally it's not computationally expensive or time-consuming. For these reasons it is included within the Pipeline object.


In [3]:
def check_decimals(val, threshold=3):
    """Evaluate the number of decimal places in a numeric value.

    Checks if the string representation of a value has more decimal places than
    a specified threshold. If it does, it returns np.nan to effectively filter
    out excessive precision, otherwise returning the original value.

    Args:
        val (int, float, str): The numeric value or its string representation to
          check.
        threshold (int, optional): The maximum allowed number of decimal places.
          Defaults to 3.

    Returns:
        float or original type: The original value, or np.nan if the threshold
        is exceeded.
    """

    # Convert to string and split by the decimal point
    str_val = str(val)
    if "." in str_val:
        decimals = str_val.split(".")[1]
        if len(decimals) > threshold:
            return np.nan
    return val


In [4]:
class DataCleaner(BaseEstimator, TransformerMixin):
    """Evaluate and clean the raw donor dataset features.

    Preprocesses the raw dataset by removing irrelevant tracking columns,
    handling structural placeholders, validating data bounds for categorical
    and rating variables.

    Args:
        categorical_cols_values (dict): A dictionary mapping categorical
          column names to lists of their admissible values.
    """

    def __init__(self, categorical_cols_values: dict):
        self.categorical_cols_values = categorical_cols_values
        self.feature_names_in_ = []

    def fit(self, X, y=None):
        """Fit the transformer on the dataset by mapping input feature headers.

        Args:
            X (pd.DataFrame): The input dataset to fit.
            y (pd.Series, optional): The target labels. Defaults to None.

        Returns:
            self: Returns the instance itself.
        """
        self.feature_names_in = []
        return self

    def transform(self, X):
        """Transform and sanitize the features of a dataset.

        Drops structural identifiers, checks and masks precision outliers,
        applies logic transformations on flags, filters out-of-bound ordinal
        values, and ensures appropriate numeric types.

        Args:
            X (pd.DataFrame): The dataset to clean.

        Returns:
            pd.DataFrame: The processed DataFrame without anomalous entries.
        """
        self.feature_names_in_ = np.array(X.columns)
        # Create a copy of the array as a precaution
        X = X.copy()
        # Drop the CONTROL_NUMBER column, as it's an identifier and, thus
        # should never be included in a model
        X.drop(["CONTROL_NUMBER"], axis=1, inplace=True)
        # Replace '?' values with a missing values
        X = X.replace("?", np.nan)
        # Set SES as a float datatype, since it was a string datatype
        # before as it contained '?' values and it contains missing values
        # which the default pandas integer datatypes don't accept.
        X["SES"] = X["SES"].astype("float")
        # Select numerical columns so that specific transformations can be
        # performed on those columns
        numerical_cols = X.select_dtypes(include=np.number).columns
        # Apply the check_decimals function to the numerical columns tp
        # transform the long decimal incorrect values into missing values
        # and then use the absolute value to flip the sign of any remanining
        # missing values, as from our exploration these appeared to be valid
        # values where the sign was simply flipped
        X[numerical_cols] = X[numerical_cols].map(check_decimals)
        X[numerical_cols] = X[numerical_cols].abs()
        # Transform the 'HOME_OWNER' variable into a binary flag column
        X["HOME_OWNER"] = X["HOME_OWNER"].apply(lambda x: 1 if x == "H" else 0)
        # Change all of the (remaining) values which are greater than 1
        # in RECENT_STAR_STATUS to 1
        X.loc[(X["RECENT_STAR_STATUS"] > 1), "RECENT_STAR_STATUS"] = 1
        # Check the categorical columns and change any unexpected value
        # to a missing value
        for var, values in self.categorical_cols_values.items():
            X.loc[(~X[var].isin(values)), var] = np.nan
        return X

    # Adding this allows set_output to work
    def get_feature_names_out(self, input_features=None):
        """Determine output features remaining after data cleaning.

        Args:
            input_features (list-like, optional): Input features descriptor.
              Defaults to None.

        Returns:
            numpy.ndarray: An array of feature names with 'CONTROL_NUMBER'
            removed.
        """
        return np.array([col for col in self.feature_names_in_ if col !=
                         "CONTROL_NUMBER"])

Since, as mentioned above, the data cleaner is stateless and has no parameters to test, but does have the parameter for admissable categorical value, we're also going to instatiate it and export it, to save the trouble of instatiating in every model notebook. For that we'll use a dictionary of admisable values for the categorical variables shown below, based on the description and the exploration we performed of the data.

In [5]:
categorical_admissible_values = {
    "DONOR_GENDER": ["M", "F", "U"],
    "PEP_STAR": [0, 1],
    "RECENCY_STATUS_96NK": ["S", "A", "E", "F", "N", "L"],
    "RECENT_STAR_STATUS": [0, 1],
    "SES": np.arange(1, 6),
    "URBANICITY": ["S", "T", "U", "R", "C"],
    "INCOME_GROUP" : np.arange(1, 8),
    "WEALTH_RATING" : np.arange(0, 10)
}

In [6]:
data_cleaner = DataCleaner(categorical_cols_values=categorical_admissible_values)
with open('Files/Pickle Files/data_cleaner.pkl', 'wb') as file:
    pickle.dump(data_cleaner, file)

# OutlierClipper

The custom OutlierClipper transformer is our tool to handle outliers effectively and allow us to test various outlier clipping strategies when applying models. This transformer has three possible methods of operation, one automatic and two which depend on the use of a mapping of bounds:
+ The automatic method uses the Interquartile Range method with a modifilable multiplier to automatically compute clipping thresholds.
+ The percentile mapping method uses a mapping of lower and upper percentiles of where the data should be clipped and then computes the thresholds from those percentiles.
+ Finally the static/"hard-coded" method uses a mapping of lower and upper values of where to clip the data exactly.

In [7]:
class OutlierClipper(BaseEstimator, TransformerMixin):
    """Clip out-of-bounds continuous outliers to specified structural limits.

    Caps extreme anomalies at the higher and lower tails of numeric feature
    distributions. It supports automated calculation via the Interquartile
    Range (IQR) approach, statistical quantile thresholds, or hardcoded
    boundary constraints passed through configuration rules.

    Args:
        rules (dict, optional): Mapping configuration utilized when `method` is
          set to 'percentile' or static rules. The keys correspond to column
          names and values represent inner dictionaries specifying 'lower' and
          'upper' target conditions. Defaults to None.
        method (str, optional): The strategy variant used to identify bounds.
          Options are 'iqr', 'percentile', or custom literal settings.
          Defaults to 'iqr'.
        iqr_multiplier (float, optional): Scaling coefficient applied to the
          Interquartile Range for determining IQR boundaries. Defaults to 1.5.
    """

    def __init__(
        self,
        rules: dict = None,
        method: str = "iqr",
        iqr_multiplier: float = 1.5,
    ):
        self.method = method
        self.rules = rules
        self.learned_limits_ = {}
        self.iqr_multiplier = iqr_multiplier
        self.feature_names_in_ = []

    def fit(self, X, y=None):
        """Fit the transformer by establishing clipping parameters for
        variables.

        Computes the target lower and upper containment values based on the
        selected strategy variant and records them within an internal
        dictionary state.

        Args:
            X (pd.DataFrame): Training matrix containing features to cap.
            y (pd.Series, optional): Target labels. Defaults to None.

        Returns:
            self: Returns the instance itself.
        """
        self.feature_names_in_ = np.array(X.columns)
        # If the Method for cliiping is the Interquartile range method
        # Compute the Quartiles, the IQR and then calculate the limits based
        # on the multiplier defined when instatiating the clipper.
        if self.method.lower() == "iqr":
            Q1 = X.quantile(0.25)
            Q3 = X.quantile(0.75)
            IQR = Q3 - Q1
            lower_threshold = Q1 - self.iqr_multiplier * IQR
            upper_threshold = Q3 + self.iqr_multiplier * IQR
            self.learned_limits_ = (
                pd.merge(
                    lower_threshold.to_frame("lower"),
                    upper_threshold.to_frame("upper"),
                    left_index=True,
                    right_index=True,
                )
                .to_dict(orient="index")
            )
        # If another method is selected use the rules dictionary to compute
        # the limits
        else:
            for var, rules in self.rules.items():
                lower_rule = rules.get("lower")
                upper_rule = rules.get("upper")
                # If percentile method is elected obtain the value that
                # corresponds to that given percentile.
                if self.method.lower() == "percentile":
                    lower_limit = (
                        X[var].quantile(lower_rule)
                        if lower_rule is not None
                        else None
                    )
                    upper_limit = (
                        X[var].quantile(upper_rule)
                        if upper_rule is not None
                        else None
                    )
                    self.learned_limits_[var] = {
                        "lower": lower_limit,
                        "upper": upper_limit,
                    }
                # If not simply take the rules from the dictionary.
                else:
                    self.learned_limits_[var] = {
                        "lower": lower_rule,
                        "upper": upper_rule,
                    }
        return self

    def transform(self, X):
        """Apply learned upper and lower constraints onto features.

        Args:
            X (pd.DataFrame): Data matrix containing elements to cap.

        Returns:
            pd.DataFrame: A modified dataset copy containing clipped numeric
              values.
        """
        X = X.copy()
        # Apply the limits to the data
        for var, limits in self.learned_limits_.items():
            # Round the limits for integer columns to maintain
            # consistency
            if X[var].dtype == int or X[var].dtype == "Int64":
                lower = np.round(limits.get("lower"))
                upper = np.round(limits.get("upper"))
            else:
                lower = limits.get("lower")
                upper = limits.get("upper")
            X[var] = X[var].clip(lower=lower, upper=upper)
        return X

    # Adding this allows set_output to work
    def get_feature_names_out(self, input_features=None):
        """Determine output features remaining after outlier transformation.

        Returns:
            numpy.ndarray: An array matching the internal input structure.
        """
        return self.feature_names_in_

# FeatureEngineer

Within the FeatureEngineer transformer we will create new feature to try and provide our model with more information which allows models and methods more information to select and utilize. The features created here and how they can be relevant were explained in the 00_EDA.ipynb notebook.T This transformer, as you'll see in the pipeline explaining section is designed before imputation, as such it is expected to propagate NaNs and let the imputer handle them. Below the section from the EDA notebook is repeated for context. Within this transformer situations of division-by-zero operations are replaced with missing values using np.where() to prevent mathematical infinity errors.

## Monetary Ratios
* **LIFETIME_AVG_GIFT_AMT**
  * Formula: `LIFETIME_GIFT_AMOUNT / LIFETIME_GIFT_COUNT`
  * Value: Standardizes total contributions against overall transaction frequency. This establishes a baseline for a donor's typical transaction size, removing distortions caused by individuals who have given many small donations over a long period.
* **LIFETIME_GIFT_AMT_RANGE**
  * Formula: `LIFETIME_MAX_GIFT_AMT - LIFETIME_MIN_GIFT_AMT`
  * Value: Measures the spread between a donor's largest and smallest historical gifts. A low range shows highly predictable, systematic donation amounts, while a high range highlights volatile or event-driven giving behavior.
* **AVG_TO_LAST_GIFT_RATIO**
  * Formula: `LIFETIME_AVG_GIFT_AMT / LAST_GIFT_AMT`
  * Value: Compares a donor's historical norm directly to their most recent transaction. A value well below 1.0 indicates that the last gift was significantly larger than their historical average, marking a positive shift in donor engagement. 

## Temporal Metrics
* **DONOR_LIFESPAN_MONTHS**
  * Formula: `MONTHS_SINCE_FIRST_GIFT - MONTHS_SINCE_LAST_GIFT`
  * Value: Measures the length of the active relationship window in months. This allows the model to distinguish between long-term consistent supporters and older, lapsed donors with a long historical tenure.
* **GIFTS_PER_MONTH_LIFESPAN**
  * Formula: `LIFETIME_GIFT_COUNT / DONOR_LIFESPAN_MONTHS`
  * Value: Measures donation frequency relative to the donor's active lifespan. This helps identify high-velocity supporters, differentiating someone who gave 5 times within a compressed 5-month period from someone who gave 5 times over 10 years.

## Socio-Economic Proxies
* **GIFT_TO_HOUSEHOLD_INCOME_RATIO**
  * Formula: `LIFETIME_AVG_GIFT_AMT / MEDIAN_HOUSEHOLD_INCOME`
  * Value: Compares a donor's average donation against the localized median income of their neighborhood. This serves as a proxy for relative generosity. For instance, a $20 average gift from an individual in a lower-income area represents a higher relative financial effort and commitment than the same $20 gift from an affluent neighborhood.

Much like the DataCleaner this class is not transferable to other problems, but yet again it is dependent on erroneous data no longer present, as such it must incoporated within the pipeline. In this case since there are no parameters, there's no point in exporting an instance of the the transformer.

In [8]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    A custom scikit-learn transformer for donor feature engineering.

    This class calculates interaction features for donor metrics while
    ensuring missing source values propagate naturally as NaN.
    """
    def __init__(self):
        self.feature_names_in_ = None
        self.engineered_features_ = None

    def fit(self, X, y=None):
        """
        Saves the input column names and sets up the list of new features.
        """
        self.feature_names_in_ = np.array(X.columns, dtype=object)
        self.engineered_features_ = [
            'LIFETIME_AVG_GIFT_AMT',
            'LIFETIME_GIFT_AMT_RANGE',
            'AVG_TO_LAST_GIFT_RATIO',
            'DONOR_LIFESPAN_MONTHS',
            'GIFTS_PER_MONTH_LIFESPAN',
            'GIFT_TO_HOUSEHOLD_INCOME_RATIO'
        ]
        return self

    def transform(self, X):
        """
        Computes the new variables. Division operations include a safeguard
        to replace 0 values with NaN to avoid infinity values.
        """
        X = X.copy()
        # Calculating the engineered features
        # We use the np.wheres to avoid mathematical problems infinity
        # erros coming from situations of division-by-zero

        # Average gift amount per transaction
        X["LIFETIME_AVG_GIFT_AMT"] = np.where(
            X["LIFETIME_GIFT_COUNT"] == 0,
            np.nan,
            X["LIFETIME_GIFT_AMOUNT"] / X["LIFETIME_GIFT_COUNT"],
        )

        # Value range between a donor's maximum and minimum gifts
        X["LIFETIME_GIFT_AMT_RANGE"] = (
            X["LIFETIME_MAX_GIFT_AMT"] - X["LIFETIME_MIN_GIFT_AMT"]
        )

        # Ratio of a donor's average gift to their very last gift
        X["AVG_TO_LAST_GIFT_RATIO"] = np.where(
            X["LAST_GIFT_AMT"] == 0,
            np.nan,
            X["LIFETIME_AVG_GIFT_AMT"] / X["LAST_GIFT_AMT"],
        )

        # Total span of donor activity in months
        X["DONOR_LIFESPAN_MONTHS"] = (
            X["MONTHS_SINCE_FIRST_GIFT"] - X["MONTHS_SINCE_LAST_GIFT"]
        )

        # Frequency of gifts relative to the donor's lifespan
        X["GIFTS_PER_MONTH_LIFESPAN"] = np.where(
            X["DONOR_LIFESPAN_MONTHS"] == 0,
            np.nan,
            X["LIFETIME_GIFT_COUNT"] / X["DONOR_LIFESPAN_MONTHS"],
        )

        # Average gift size relative to local neighborhood income
        X["GIFT_TO_HOUSEHOLD_INCOME_RATIO"] = np.where(
            X["MEDIAN_HOUSEHOLD_INCOME"] == 0,
            np.nan,
            X["LIFETIME_AVG_GIFT_AMT"] / X["MEDIAN_HOUSEHOLD_INCOME"],
        )
        return X

    def get_feature_names_out(self, input_features=None):
        """
        Returns the combined list of original and engineered feature names.
        """
        return np.append(self.feature_names_in_, self.engineered_features_)

# CategoricalFeatureSelector

The CategoricalFeatureSelector class is very simple in its function, it receives a dataset and produces a contingency table of frequencies, which it then uses to perform a chi-squared test, it will then select relevant variables based on a modifiable p-value threshold, which by default is set to the conventional value of 0.05. This class allows to effectively integrate a chi-squared indepedence for our categorical features independently from the feature selection tests that will be performed for numerical features. <p> The chi-squared is a great way to test whether a categorical predictor is relevant, by evaluating independence to the target, and thus a valid predictor or not. 

In [9]:
class CategoricalFeatureSelector(BaseEstimator, TransformerMixin):
    """Select categorical features based on a chi-squared test of independence.

    Evaluates the association between each input categorical feature and a
    target variable using a contingency table and a chi-squared test. Features
    are statistically selected and retained only if their computed p-value
    falls below a designated significance threshold.

    Args:
        p_value (float, optional): The statistical significance threshold
          (alpha level) used to reject the null hypothesis of independence.
          Features with a p-value strictly less than this threshold are
          retained. Defaults to 0.05.
    """

    def __init__(self, p_value: float = 0.05):
        self.p_value = p_value
        self.cols_to_keep_ = []

    def fit(self, X, y):
        """Fit the selector by evaluating feature associations with the target.

        Iterates through each column in the dataset, constructs a cross
        -tabulation contingency table with the target variable, performs a
        chi-squared test,and saves the name of the feature if it shows a
        statistically significant association.

        Args:
            X (pd.DataFrame): The input dataset containing categorical
              features.
            y (pd.Series or array-like): The binary or categorical target
              labels used to compute associations.

        Returns:
            self: Returns the instance itself.
        """
        # Initating the columns to keep to ensure no information
        # is leaked from previous folds
        self.cols_to_keep_ = []

        for var in X.columns:
            # Perform the chi-squared test and keep the variable if the
            # test p-value is lower than the threshold and thus the null
            # hypothesis is rejected indicating an association between the
            # variable and the target.
            if chi2_contingency(pd.crosstab(y, X[var]))[1] < self.p_value:
                self.cols_to_keep_.append(var)
        return self

    def transform(self, X):
        """Slice the input dataset to retain only statistically chosen
        features.

        Args:
            X (pd.DataFrame): The dataset to perform feature reduction on.

        Returns:
            pd.DataFrame: A subset of the input DataFrame consisting only of
              the statistically significant features determined during fitting.
        """
        return X[self.cols_to_keep_]

    def get_feature_names_out(self, input_features=None):
        """Determine output features remaining after feature selection.

        Args:
            input_features (list-like, optional): Input features descriptor.
              Defaults to None.

        Returns:
            numpy.ndarray: An array of strings containing the names of the
              retained features.
        """
        return np.array(self.cols_to_keep_)

# NumericalFeatureSelector

The NumeicalFeatureSelector transformer aims to go beyond simple filter methods for the purposes of feature selection. To achieve this, within 3 separate tests are performed and a voting system is used:
+ Firstly a test based on spearman correlation where for each highly correlated feature pair, the feature with the lowest correlation to the target gets a removal vote.
+ Then an Recursive Feature Elimination test is performed using a simple tree model to reduce complexity within an RFECV object to ensure that the best number of features is selected, the features that aren't selected receive a vote for removal.
+ Finally, a Lasso (LogisticRegression with L1 regularization, since this is a classification problem and the Lasso object in scikit-learn is a regressor) test is performed and any feature with a coefficient lower than 0.00001 receive a vote for removal.

Then, any variable with 2 or more votes for removal is excluded, leaving only variables with 1 vote or no votes as the selected variables.


In [ ]:
class NumericalFeatureSelector(BaseEstimator, TransformerMixin):
    """Select numerical features using an ensemble ensemble-voting strategy.

    Combines three distinct feature screening paradigms to dynamically rate and
    prune redundant or low-value continuous variables:
    1. Spearman rank correlation to eliminate collinearity.
    2. Cross-validated Recursive Feature Elimination (RFECV) using a
       Decision Tree to isolate structural importance.
    3. L1-regularized Logistic Regression (Lasso) to enforce absolute sparsity.

    Features accumulate "removal votes" across all three assessments and are
    dropped only if flagged by a majority (2 or more) of the methods.

    Args:
        corr_threshold (float, optional): The absolute Spearman correlation
          cutoff point. Pairwise variables exceeding this bound will trigger a
          removal vote for the column carrying lower target affinity. Defaults
          to 0.8.
        cv (scikit-learn splitter, optional): Cross-validation generator used
          to safely stratify evaluations across internal selectors. Defaults to
          StratifiedKFold(n_splits=5, shuffle=True, random_state=23).
        random_state (int, optional): Control seed injected into stochastic
          underlying estimators to yield deterministic feature outcomes.
          Defaults to 42.
    """

    def __init__(
        self,
        corr_threshold: float = 0.8,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=23),
        random_state: int = 42,
    ):
        self.corr_threshold = corr_threshold
        self.cv = cv
        self.random_state = random_state
        self.cols_to_keep_ = []

    def fit(self, X, y):
        """Fit the selector by gathering ensemble votes across numeric
        variables.

        Runs Spearman correlation filtering, cross-validated RFE tree splits,
        and L1 logistic regularizations. Features with 2 or more accumulated
        votes for deletion are pruned from the downstream execution list.

        Args:
            X (pd.DataFrame): Input numerical features training matrix.
            y (pd.Series or array-like): Target classification labels.

        Returns:
            self: Returns the instance itself.
        """
        # Initating the columns to keep and the votes
        # to ensure no information is leaked from
        # previous folds
        votes_to_remove = {col: 0 for col in X.columns}

        # Method 1 - Correlation Based feature selection
        # If two features are considered highly correlated then the feature
        # with the lowest correlation with the target is voted to be removed
        # We use spearman correlation since it captures non-linear correlations

        # Getting the absolute value of the correlations, since for this step
        # only the magnitude is relevant and not the sign
        corr_matrix = X.corr(method="spearman").abs()
        # Getting the correlation of features with the target
        target_corr = X.apply(lambda x: x.corr(y, method="spearman")).abs()
        # Filtering the correlation matrix to include only the upper triangle
        # so that each feature is only tested once
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        for col in upper.columns:
            # Finding rows where correlation is above the defined threshold
            high_corr_cols = upper.index[
                upper[col] > self.corr_threshold
                ].tolist()
            for row in high_corr_cols:
                # Voting to remove the feature with lower correlation with the
                # target
                to_drop = col if target_corr[col] < target_corr[row] else row
                # Ensuring that clusters of highly correlated columns don't
                # result in extreme amounts of votes, by essentially capping
                # votes at 1 for this test.
                if votes_to_remove[to_drop] == 0:
                    votes_to_remove[to_drop] += 1

        # Method 2 - Simple Recursive Feature Elimination
        # We use a simple Decision Tree model to avoid drastically increasing
        # the running time To increase robustness we use a crossvalidated RFE,
        # this will test the best number of features using cross validation,
        # leading to a much more robust outcome.

        # Creating and fitting the RFE object
        rfe = RFECV(
            estimator=DecisionTreeClassifier(max_depth=5,
                                             class_weight="balanced",
                                             random_state=self.random_state),
            cv=self.cv,
            scoring="f1",
            n_jobs=-1)
        rfe.fit(X, y)
        for col, keep in zip(X.columns, rfe.support_):
            # Voting to remove the feature if RFE doesn't consider it worth
            # keeping
            if not keep:
                votes_to_remove[col] += 1

        # Method 3 - Lasso
        # Note: We use LogisticRegression with L1 because TARGET_B is binary
        # This method uses L1 regularization to reduce coeficients of low-value
        # predictors to values close to 0

        # Creating and fitting the RFE object
        lasso = LogisticRegressionCV(
            cv=self.cv,
            penalty="l1",
            solver="liblinear",
            max_iter=1000,
            scoring="f1",
            random_state=self.random_state,
            n_jobs=-1
        ).fit(X, y)

        for col, coef in zip(X.columns, lasso.coef_[0]):
            # Voting to remove the feature if its Lasso coefficient
            # is below 0.00001
            if abs(coef) < 0.00001:
                votes_to_remove[col] += 1

        # Since we're only running 3 tests, we're going to keep any variable
        # that has 1 or no votes for removal, meaning that any variables with
        # 2 or more votes is excluded
        self.cols_to_keep_ = [
            col
            for col, n_votes_to_remove in votes_to_remove.items()
            if n_votes_to_remove <= 1
        ]
        return self

    def transform(self, X):
        """Slice the input dataset to keep variables with a passing vote score.

        Args:
            X (pd.DataFrame): The dataset matrix to filter.

        Returns:
            pd.DataFrame: Sliced feature matrix holding only non-voted out
              columns.
        """
        return X[self.cols_to_keep_]

    def get_feature_names_out(self, input_features=None):
        """Determine output features remaining after ensemble voting reduction.

        Args:
            input_features (list-like, optional): Input features descriptor.
              Defaults to None.

        Returns:
            numpy.ndarray: An array of retained feature name strings.
        """
        return np.array(self.cols_to_keep_)

# Final Pipeline architechure

Our final pipeline starts with the custom DataCleaner transformer to clean data of clear erroneous data before more advanced preprocessing can begin, it is then divided into two sub-pipelines that perform different data preprocessing:
+ One sub-pipeline handles categorical data, within it mode imputation, feature selection with our custom transformer and encoding is performed.
+ The other sub-pipeline handles numerical data, in this sub-pipeline outliers can be clipped, new features can created, then features are scaled and imputed and finally our custom feature selector selects numerical features. For now, we have RobustScaler and KNNimputer as the scaler and the imputer, but we will test other options.

Finally, the last step of our pipeline is the model used to make predictions. In this step we are going the TunedThresholdClassifierCV wrapped around the models we're testing and using, with this being an unbalanced binary classification problem where the primary metric is F1 score, optimizing the threshold can be a very powerful tool to extract better performances out of seemingly low-quality models. We dont use pass a custom cv method into it, because the default is a 5-fold StratifiedKFold, which is what we're using for model testing anyway.

Note: The tree model currently present is simply a placeholder.


In [11]:
# Categorical Feature Sub-Pipeline
# This is the part that handles the categorical columns, performing
# mode imputation, feature selection using our custom CategoricalFeatureSelector
# and finally one-hot encoding the features.
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('feature_selection',  CategoricalFeatureSelector()),
    # 3. Your specialized encoding (OneHot/Target) now receives imputed integers
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')),
])

# Numerical Feature Sub-Pipeline
# Here we take care of our numerical features, starting with outlier clipping and feature
# creation using our custom transformer, then scaling data, so that it can then 
# be imputed and numerical feature selection can be performed
num_pipe = Pipeline([
    ('clipper', OutlierClipper()),
    ('feature_engineer', FeatureEngineer()),
    ('scaler', RobustScaler()),
    ('imputer', KNNImputer()),
    ('feature_selection', NumericalFeatureSelector(random_state=42))
])

# Here we use a ColumnTransformer with column selectors to perform the split
# between numerical and categorical data, so that each subset can be directed
# to the appropriate sub-pipeline
preprocessor = ColumnTransformer([
    ('cat_section', cat_pipe, make_column_selector(dtype_exclude=[np.number])),
    ('num_section', num_pipe, make_column_selector(dtype_include=[np.number])),
],
verbose_feature_names_out=False)
#  

 
# Final Pipeline
pipe = Pipeline([
    ('cleaner', DataCleaner(categorical_cols_values=categorical_admissible_values)),
    ('preprocessing', preprocessor),
    ('model', TunedThresholdClassifierCV(DecisionTreeClassifier(),
                                         n_jobs=-1,
                                         scoring='f1',
                                         random_state=42
                                         )) 
])

# We use the set_output to pandas so that intermediate transformers can use column
# names, since some of the default scikit-learn transformers by default return 
# Numpy arrays, which obviously don't have column names.
pipe.set_output(transform="pandas")

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cleaner', ...), ('preprocessing', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,categorical_cols_values,"{'DONOR_GENDER': ['M', 'F', ...], 'INCOME_GROUP': array([1, 2, 3, 4, 5, 6, 7]), 'PEP_STAR': [0, 1], 'RECENCY_STATUS_96NK': ['S', 'A', ...], ...}"
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_section', ...), ('num_section', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that

As mentioned in the General explanataion of our modeling process, we will use a stratified hold-out method to look for parameter regions to test, as such, as the last part of this notebook we're going to import the training data, split it into training and validation run it through the pipeline (excluding the model) to get preprocessed versions of those partitions, which we'll then export using pickle. We'll use 20% as validation since we have a decently sized dataset and 20% of it is over 2500 rows which we believe is good enough for this purpose, whilst allowing a little extra data for training when compared to using 25% or 30%.

In [12]:

train = pd.read_csv('Files/donors_train.csv')

In [13]:
X = train.drop(['TARGET_B'], axis=1)
y = train['TARGET_B']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=True, stratify=y, random_state=42)

In [14]:
preprocessor = clone(pipe[:-1])
X_train_preprocessed = preprocessor.fit_transform(X_train, y_train)
X_val_preprocessed = preprocessor.transform(X_val)

In [15]:
X_train_preprocessed.to_pickle('Files/Pickle Files/X_train_preprocessed.pkl')
y_train.to_pickle('Files/Pickle Files/y_train.pkl')
X_val_preprocessed.to_pickle('Files/Pickle Files/X_val_preprocessed.pkl')
y_val.to_pickle('Files/Pickle Files/y_val.pkl')

# Parameter testing function

The last tool we developed was a function to perform paramter searching. The reason we developed our own instead of using sklearn's built-in options is that with our custom function we can use the tqdm library to track progresssion. Our function performs cross validation, and has options for exporting the results dataframe and well as refitting the best performing model and exporting that too, using pickle. The results are overwritten with every parameter combination to keep records even if something goes wrong. 

In [ ]:
def run_parameter_search(grid: dict,
                         cv: any,
                         X: any,
                         y: any,
                         model: any,
                         metrics: list,
                         results_file_dir: str = None,
                         model_file_dir: str = None,
                         refit: bool = True,
                         n_jobs: int = None) -> pd.DataFrame:
    """Performs a manual parameter search with cross-validation.

    This allows the inclusion of a TQDM progress bar to track hyperparameter
    tuning and evaluation progress.

    Args:
        grid (dict): Dictionary with parameter names as keys and lists of
            settings to try as values.
        cv (any): Determines the cross-validation splitting strategy (e.g., an
            integer or a scikit-learn CV splitter).
        X (any): The training input samples.
        y (any): The target values.
        model (any): The estimator object to use to fit the data.
        metrics (list): The list of metric functions or names to be used for
            evaluating the model. The first metric is the one used to sort
            results at the end.
        results_file_dir (str, optional): Directory path where the search
            results DataFrame will be saved. Defaults to None.
        model_file_dir (str, optional): Directory path where the best fitted
            model. Defaults to None.
        refit (bool, optional): If True, refits the best estimator using the
            entire dataset. Defaults to True.

    Returns:
        pd.DataFrame: A DataFrame containing the parameter combinations,
          mean/std scores for training and validation splits, and execution
          status.
    """
    # Build a ParameterGrid based on the parameters to test
    params = ParameterGrid(grid)
    # Initializing the results and the primary metric, which will
    # be the one used to sort runs at the end.
    results = []
    primary_metric = metrics[0]
    for param in tqdm(params, desc="Tuning Hyperparameters"):
        # Initializing the run record with the full dictionary of parameters
        # as well as unpacking that dictionary to create a column for each
        # parameter as that may be useful.
        run_record = {'params_config': param.copy(),
                      **param}
        # We use the try/except block to ensure that if there's any issue with
        # one parameter combination (certain parameter settings are
        # incompatible) we still test all valid ones
        try:
            # Cloning the input model, applying the paramters
            # and the pandas output mode
            current_model = clone(model)
            current_model.set_params(**param)
            current_model.set_output(transform="pandas")
            # Running the cross-validation and processing
            # all of the relevant metrics (scores and fit-time)
            crossval_results = cross_validate(current_model,
                                              X,
                                              y,
                                              cv=cv,
                                              return_train_score=True,
                                              scoring=metrics,
                                              n_jobs=n_jobs,
                                              error_score='raise')
            run_record['mean_fit_time'] = np.mean(crossval_results['fit_time'])
            for metric in metrics:
                run_record[f'mean_val_{metric}'] = np.mean(
                    crossval_results[f'test_{metric}'])
                run_record[f'std_val_{metric}'] = np.std(
                    crossval_results[f'test_{metric}'])
                run_record[f'mean_train_{metric}'] = np.mean(
                    crossval_results[f'train_{metric}'])
                run_record[f'std_train_{metric}'] = np.std(
                    crossval_results[f'train_{metric}'])
            # Classifying the combination as successfull
            run_record['status'] = 'Success'
        except Exception as e:
            # If the parameter combination fails put missing values in all
            # of the metric columns
            run_record['mean_fit_time'] = np.nan
            for metric in metrics:
                run_record[f'mean_val_{metric}'] = np.nan
                run_record[f'std_val_{metric}'] = np.nan
                run_record[f'mean_train_{metric}'] = np.nan
                run_record[f'std_train_{metric}'] = np.nan
            # Classifying the combination as failed
            run_record['status'] = f'Failed: {str(e)}'
        # Appending the run_record to the results
        results.append(run_record)
        # Creating the dataframe and sorting by the primary metric
        df_result = pd.DataFrame(results).sort_values(
            f'mean_val_{primary_metric}', ascending=False, na_position='last')
        # Checking whether there is a directory to export results
        # and using pickle to export it if there is
        if results_file_dir is not None:
            df_result.to_pickle(results_file_dir)

    # Checking whether the best model is to be refitted on the full data
    # and grabbing the best parameter combination, applying it and
    # fitting the model if it is to be refitted.
    # Only do it if the best score comes from a successful run
    # In other words, if there is at least one succesfull run
    if refit and df_result.iloc[0]['status'] == 'Success':
        best_model = clone(model)
        best_model.set_params(**df_result.iloc[0]['params_config'])
        best_model.set_output(transform="pandas")
        best_model_fitted = best_model.fit(X, y)
        # Checking whether there is a directory to export the model
        # and using pickle to export it if there is
        if model_file_dir is not None:
            with open(model_file_dir, 'wb') as file:
                pickle.dump(best_model_fitted, file)
        return df_result, best_model_fitted
    else:
        return df_result
